In [1]:
import numpy as np   
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.version.cuda)
print(f"running on {device}")

13.0
running on cuda


In [2]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, dtype = torch.float32)
model.to(device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [8]:
system_prompt = """
    You are an agentic language model built to use tools to answer questions. You have the following tools right now:
    - get_time(): query the current time
    - query_notes_database(query): queries the user's home database and returns the details and file path
    - read_file(path): reads the content of the file at "path", if it doesn't exist it says so
    - write_file(path, content): adds the "content" to the end of "path" if it exists, if not, creates file

    The user will give a task. Your job is to review the tools available and return ONLY the function call signature
"""

def format_prompt(user, system = system_prompt):
    return (f"""<|system|> 
            {system} </s> 
            <|user|> 
            {user}  
            <\s>
            <|assistant|>\n""")


model.eval()
inputs = tokenizer(format_prompt("add the notes from today's econ lecture to the most applicable file"), return_tensors="pt").to(device)
output_ids = model.generate(
    **inputs,
    max_new_tokens=500,
    do_sample=True,
    temperature=0.8,
    top_p=0.9
)

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

Both `max_new_tokens` (=500) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|system|> 
            
    You are an agentic language model built to use tools to answer questions. You have the following tools right now:
    - get_time(): query the current time
    - query_notes_database(query): queries the user's home database and returns the details and file path
    - read_file(path): reads the content of the file at "path", if it doesn't exist it says so
    - write_file(path, content): adds the "content" to the end of "path" if it exists, if not, creates file

    The user will give a task. Your job is to review the tools available and return ONLY the function call signature
  
            <|user|> 
            add the notes from today's econ lecture to the most applicable file  
            <\s>
            <|assistant|>
            Please provide the details and instructions for adding the notes from today's econ lecture to the most applicable file, as mentioned in the prompt. Instructions for adding notes to a file will be provided based on the specific 

In [ ]:
import json
import torch
import re
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW

# ── Config ────────────────────────────────────────────────────────────────────

MODEL_NAME   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATA_PATH    = "gentool_synthetic.json"
SAVE_PATH    = "./gentool-finetuned"
MAX_LEN      = 768
BATCH_SIZE   = 4
LR           = 2e-5
EPOCHS_S1    = 3   # ranking stage
EPOCHS_S2    = 3   # selection stage
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

# ── Tool formatting ───────────────────────────────────────────────────────────

GENERATE_RESPONSE_BLOCK = (
    "generate_response()\n"
    "  Use this when no available tool is suitable for the query."
)

def format_tool(tool: dict) -> str:
    params_sig = ", ".join(
        f"{p['name']}: {p['type']}" for p in tool["parameters"]
    )
    param_details = "\n".join(
        f"  {p['name']}: {p['description']}" for p in tool["parameters"]
    )
    return (
        f"{tool['name']}({params_sig})\n"
        f"  {tool['description']}\n"
        f"{param_details}"
    )

def format_toolbox(tools: list) -> str:
    blocks = [format_tool(t) for t in tools]
    blocks.append(GENERATE_RESPONSE_BLOCK)
    return "\n\n".join(blocks)

# ── Prompt builders ───────────────────────────────────────────────────────────

SYSTEM = (
    "You are a tool-calling assistant. "
    "Read each tool description carefully. "
    "Select the most suitable tool for the query. "
    "If no tool is suitable, output generate_response()."
)

def build_ranking_prompt(sample: dict) -> str:
    toolbox = format_toolbox(sample["tools"])
    return (
        f"<|system|>\n{SYSTEM}</s>\n"
        f"<|user|>\n"
        f"Query: {sample['query']}\n\n"
        f"Available tools:\n\n{toolbox}\n\n"
        f"Rank all tools from most to least useful for this query. "
        f"Evaluate every tool explicitly.</s>\n"
        f"<|assistant|>\n"
    )

def build_ranking_answer(sample: dict) -> str:
    r = sample["ranking"]
    lines = [
        f"1st: {r['1st']}",
        f"2nd: {r['2nd']}",
        f"3rd: {r['3rd']}",
        f"4th: {r['4th']}",
        f"5th: {r['5th']}",
        f"\nReasoning: {sample['reasoning']}"
    ]
    return "\n".join(lines) + "</s>"

def build_selection_prompt(sample: dict) -> str:
    toolbox = format_toolbox(sample["tools"])
    return (
        f"<|system|>\n{SYSTEM}</s>\n"
        f"<|user|>\n"
        f"Query: {sample['query']}\n\n"
        f"Available tools:\n\n{toolbox}\n\n"
        f"Select the best tool and call it. "
        f"If no tool is suitable output generate_response().</s>\n"
        f"<|assistant|>\n"
    )

def build_selection_answer(sample: dict) -> str:
    name = sample["correct_tool"]
    args = sample["correct_arguments"]
    if name == "generate_response" or not args:
        return "generate_response()</s>"
    args_str = ", ".join(f"{k}='{v}'" for k, v in args.items())
    return f"{name}({args_str})</s>"

# ── Dataset ───────────────────────────────────────────────────────────────────

class GenToolDataset(Dataset):
    """
    stage: "ranking" or "selection"
    Each item produces (prompt, answer) where loss is computed
    only on answer tokens.
    """

    def __init__(self, samples: list, tokenizer, stage: str, max_len: int = MAX_LEN):
        self.samples   = samples
        self.tokenizer = tokenizer
        self.stage     = stage
        self.max_len   = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        if self.stage == "ranking":
            prompt = build_ranking_prompt(s)
            answer = build_ranking_answer(s)
        else:
            prompt = build_selection_prompt(s)
            answer = build_selection_answer(s)

        full_text = prompt + answer

        full_enc = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        prompt_enc = self.tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False
        )

        input_ids      = full_enc["input_ids"].squeeze()
        attention_mask = full_enc["attention_mask"].squeeze()
        labels         = input_ids.clone()

        # Mask prompt — only train on answer tokens
        prompt_len = min(prompt_enc["input_ids"].shape[1], self.max_len)
        labels[:prompt_len] = -100

        # Also mask padding
        labels[attention_mask == 0] = -100

        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         labels
        }

# ── Training loop ─────────────────────────────────────────────────────────────

def run_stage(model, dataset, epochs: int, lr: float, stage_name: str):
    loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        avg = total_loss / len(loader)
        print(f"[{stage_name}] epoch {epoch + 1}/{epochs} — loss: {avg:.4f}")

# ── Inference (for testing on Pi) ────────────────────────────────────────────

def select_tool(model, tokenizer, query: str, tools: list) -> str:
    """
    Given a query and a live tool list, run the fine-tuned model
    and return the tool call string.
    """
    sample  = {"query": query, "tools": tools}
    prompt  = build_selection_prompt(sample)
    enc     = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    model.eval()
    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=80,
            do_sample=False,          # greedy for tool selection
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = out[0][enc["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    print(f"Loading tokenizer and model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32
    )
    model.to(DEVICE)

    # Optional: INT8 quantization to cut memory
    # model = torch.quantization.quantize_dynamic(
    #     model, {torch.nn.Linear}, dtype=torch.qint8
    # )

    print(f"Loading data from {DATA_PATH}")
    with open(DATA_PATH) as f:
        samples = json.load(f)

    print(f"Loaded {len(samples)} samples")

    # ── Stage 1: Ranking ──────────────────────────────────────────────────────
    print("\n=== Stage 1: Tool Ranking ===")
    stage1 = GenToolDataset(samples, tokenizer, stage="ranking")
    run_stage(model, stage1, epochs=EPOCHS_S1, lr=LR, stage_name="ranking")

    # ── Stage 2: Selection ────────────────────────────────────────────────────
    print("\n=== Stage 2: Tool Selection ===")
    stage2 = GenToolDataset(samples, tokenizer, stage="selection")
    run_stage(model, stage2, epochs=EPOCHS_S2, lr=LR * 0.5, stage_name="selection")

    # ── Save ──────────────────────────────────────────────────────────────────
    print(f"\nSaving to {SAVE_PATH}")
    model.save_pretrained(SAVE_PATH)
    tokenizer.save_pretrained(SAVE_PATH)
    print("Done.")

    # ── Quick sanity check ────────────────────────────────────────────────────
    print("\n=== Sanity check ===")
    test_tools = samples[0]["tools"]
    test_query = samples[0]["query"]
    result = select_tool(model, tokenizer, test_query, test_tools)
    expected = build_selection_answer(samples[0]).replace("</s>", "").strip()
    print(f"Query:    {test_query}")
    print(f"Expected: {expected}")
    print(f"Got:      {result}")


if __name__ == "__main__":
    main()